# Compare Samples in Two Feature Spaces

A [`DatasetCollection`](../../python/dataset/api.md) makes samples the
observations and stores each feature space calculated for them as a modality
(see [Collections](../../python/collection/api.md)). This brief example uses a
small **simulated** cohort of 8 samples, 4 healthy and 4 inflamed, to describe
the same samples in two spaces:

- **Cell-type population space**: each sample's cell-type proportions, shown in a
  [Composition](../../visualizations/composition.md) view.
- **B cell expression signature space**: the pseudobulk expression of each
  sample's B cells, clustered in a [Clustergram](../../visualizations/clustergram.md).

In [1]:
import anndata as ad
import numpy as np
import pandas as pd

import celldega as dega

## Simulate a small cohort

Inflamed samples have more immune cells, and their B cells express a plasma-cell
program (`JCHAIN`, `MZB1`, `XBP1`) that healthy samples' B cells don't.

In [2]:
rng = np.random.default_rng(0)

cell_types = ["B cells", "T cells", "Macrophages", "Epithelial", "Fibroblasts"]
mix = {"healthy": [0.08, 0.15, 0.10, 0.47, 0.20], "inflamed": [0.20, 0.30, 0.20, 0.20, 0.10]}
genes = ["MS4A1", "CD79A", "CD19", "JCHAIN", "MZB1", "XBP1", "CD3E", "CD68", "EPCAM", "COL1A1"]
markers = {
    "B cells": ["MS4A1", "CD79A", "CD19"],
    "T cells": ["CD3E"],
    "Macrophages": ["CD68"],
    "Epithelial": ["EPCAM"],
    "Fibroblasts": ["COL1A1"],
}
plasma_program = ["JCHAIN", "MZB1", "XBP1"]

obs, counts = [], []
for i in range(8):
    condition = "healthy" if i < 4 else "inflamed"
    proportions = rng.dirichlet(np.array(mix[condition]) * 150)
    for cell_type in rng.choice(cell_types, 400, p=proportions):
        rate = np.full(len(genes), 0.2)
        rate[[genes.index(g) for g in markers[cell_type]]] = 6.0
        if cell_type == "B cells":
            rate[[genes.index(g) for g in plasma_program]] = 4.0 if condition == "inflamed" else 0.5
        counts.append(rng.poisson(rate))
        obs.append((f"S{i + 1}", f"P{i // 2 + 1}", condition, cell_type))

adata = ad.AnnData(
    np.array(counts, dtype=float),
    obs=pd.DataFrame(
        obs,
        columns=["sample_id", "patient_id", "condition", "cell_type"],
        index=[f"cell_{i}" for i in range(len(obs))],
    ),
    var=pd.DataFrame(index=genes),
)
adata.obs["cell_type"] = pd.Categorical(adata.obs["cell_type"], categories=cell_types)
adata.uns["cell_type_colors"] = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
adata

AnnData object with n_obs × n_vars = 3200 × 10
    obs: 'sample_id', 'patient_id', 'condition', 'cell_type'
    uns: 'cell_type_colors'

## Build the collection

One row per sample, with patient and condition carried over from the cells.

In [3]:
dset = dega.DatasetCollection(
    adata, dataset_col="sample_id", obs_columns=["patient_id", "condition"]
)
dset.obs

,sample_id,n_cells,patient_id,condition
sample_id,,,,
S1,S1,400,P1,healthy
S2,S2,400,P1,healthy
S3,S3,400,P2,healthy
S4,S4,400,P2,healthy
S5,S5,400,P3,inflamed
S6,S6,400,P3,inflamed
S7,S7,400,P4,inflamed
S8,S8,400,P4,inflamed


## Cell-type population space

`calc_population` adds a samples × cell types modality. The Composition view
draws each sample as a stacked bar; click **clust** to order the samples by
their composition, and **PROP** / **COUNTS** to switch between proportions and
counts.

In [4]:
dset.calc_population(adata, category="cell_type")
dset.mod["population"].to_df().round(2)

cell_type,B cells,Epithelial,Fibroblasts,Macrophages,T cells
sample_id,,,,,
S1,0.08,0.53,0.19,0.05,0.15
S2,0.14,0.40,0.22,0.12,0.12
S3,0.06,0.53,0.24,0.09,0.08
S4,0.10,0.45,0.20,0.11,0.13
S5,0.26,0.11,0.13,0.16,0.34
S6,0.20,0.17,0.14,0.14,0.34
S7,0.21,0.22,0.09,0.21,0.28
S8,0.22,0.24,0.09,0.20,0.27


In [5]:
composition = dega.viz.Composition(
    dset, group_attrs=["condition"], adata=adata, width=600, height=300
)

In [6]:
composition

## B cell expression signature space

`calc_signature` adds the pseudobulk expression of one population per sample:
here, each sample's B cells. Clustered in a Clustergram, the samples split by
condition on the plasma-cell program, which the population space can't show.

In [7]:
dset.calc_signature(adata, category="cell_type", value="B cells")

mat = dega.clust.Matrix(dset.mod["b_cells_signature"], col_attr=["condition"])
mat.norm(axis="row", by="zscore")
mat.cluster()
cgm = dega.viz.Clustergram(matrix=mat, width=450, height=300)

In [8]:
cgm

Both spaces live on the same collection and save together:

```python
dset.write("cohort.h5mu")
dega.DatasetCollection.read("cohort.h5mu")
```